# 🌟 Star Schema Creation
## Shoebadoo Sales Analytics - Phase 3

Vorherigen Schritte findet ihr in den Notebooks 1-3 und die Ergebnisse unter dem Ordner Data und results_documentation
Nachdem, wir die Daten bereinigt und gecheckt haben, lass uns zum nächsten Schritt gehen!

---

## Was machen wir?
Ich habe ein Starschema Design entwickelt und die einzelnen Tables dokumentiert. 
Die ausgearbeiteten Markdowns inklusive Schema Diagramm findet ihr hier:

https://github.com/Fonks/erp-sales-analytics-pipeline/tree/main/data_warehouse

Nun ans eingemachte!

Wir bauen ein **Star Schema** mit PySpark:

1. **4 Dimension Tables** erstellen (Date, Customer, Product, Channel)
2. **1 Fact Table** erstellen (Sales)
3. **Alles als Parquet speichern** für Analytics

---

## 1. Setup

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, year, month, quarter, dayofmonth, dayofweek, weekofyear,
    date_format, when, monotonically_increasing_id, row_number, to_date
)
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder \
    .appName("Shoebadoo_StarSchema") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("✅ Spark Session created")
print(f"Spark Version: {spark.version}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/07 13:58:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/07 13:58:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/07 13:58:41 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


✅ Spark Session created
Spark Version: 3.5.0


## 2. Cleaned Data laden

In [2]:
INPUT_PATH = "/app/data/cleaned/2_data_cleaning"
OUTPUT_PATH = "/app/data/warehouse/3_star_schema"

sales_df = spark.read.parquet(f"{INPUT_PATH}/sales_clean.parquet")
products_df = spark.read.parquet(f"{INPUT_PATH}/products_clean.parquet")
customers_df = spark.read.parquet(f"{INPUT_PATH}/customers_clean.parquet")

print(f"✅ Sales: {sales_df.count():,} rows")
print(f"✅ Products: {products_df.count():,} rows")
print(f"✅ Customers: {customers_df.count():,} rows")
print("\n📋 Sales Schema:")
sales_df.printSchema()

✅ Sales: 397,962 rows
✅ Products: 454 rows
✅ Customers: 8,000 rows

📋 Sales Schema:
root
 |-- product_id: long (nullable = true)
 |-- sale_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- channel: string (nullable = true)
 |-- sale_datetime: timestamp_ntz (nullable = true)
 |-- quantity: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)



25/11/07 13:58:53 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## 3. DIM_DATE erstellen

**WICHTIG:** Die Spalte heißt `sale_datetime`, nicht `sale_date`!

In [3]:
# Date aus sale_datetime extrahieren
dates_df = sales_df.select(
    to_date(col("sale_datetime")).alias("full_date")
).distinct()

# Date Attributes
dim_date = dates_df.select(
    date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),
    col("full_date"),
    year("full_date").alias("year"),
    quarter("full_date").alias("quarter"),
    month("full_date").alias("month"),
    date_format("full_date", "MMMM").alias("month_name"),
    weekofyear("full_date").alias("week"),
    dayofmonth("full_date").alias("day"),
    dayofweek("full_date").alias("day_of_week"),
    date_format("full_date", "EEEE").alias("day_name"),
    when(dayofweek("full_date").isin([1, 7]), True).otherwise(False).alias("is_weekend")
).orderBy("date_key")

print(f"✅ dim_date: {dim_date.count():,} rows")
dim_date.show(5)

✅ dim_date: 1,279 rows
+--------+----------+----+-------+-----+----------+----+---+-----------+---------+----------+
|date_key| full_date|year|quarter|month|month_name|week|day|day_of_week| day_name|is_weekend|
+--------+----------+----+-------+-----+----------+----+---+-----------+---------+----------+
|20220323|2022-03-23|2022|      1|    3|     March|  12| 23|          4|Wednesday|     false|
|20220324|2022-03-24|2022|      1|    3|     March|  12| 24|          5| Thursday|     false|
|20220325|2022-03-25|2022|      1|    3|     March|  12| 25|          6|   Friday|     false|
|20220326|2022-03-26|2022|      1|    3|     March|  12| 26|          7| Saturday|      true|
|20220327|2022-03-27|2022|      1|    3|     March|  12| 27|          1|   Sunday|      true|
+--------+----------+----+-------+-----+----------+----+---+-----------+---------+----------+
only showing top 5 rows



## 4. DIM_CUSTOMER erstellen

In [4]:
window = Window.orderBy("customer_id")

dim_customer = customers_df.select(
    row_number().over(window).alias("customer_key"),
    col("customer_id"),
    col("customer_name"),
    col("customer_segment"),
    col("country"),
    col("city"),
    col("postal_code"),
    col("registration_date")
)

print(f"✅ dim_customer: {dim_customer.count():,} rows")
dim_customer.show(5)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `customer_name` cannot be resolved. Did you mean one of the following? [`customer_id`, `last_name`, `first_name`, `country`, `email`].;
'Project [row_number() windowspecdefinition(customer_id#28L ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS customer_key#164, customer_id#28L, 'customer_name, 'customer_segment, country#33, 'city, 'postal_code, registration_date#32]
+- Relation [customer_id#28L,first_name#29,last_name#30,email#31,registration_date#32,country#33,date_of_birth#34] parquet
